# 05 — Финальное решение V27

In [1]:
from __future__ import annotations

import gc
import json
import math
import shutil
import subprocess
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 42
EXPECTED_USERS = 250_000
HORIZON = 30
DEVICES = "0"

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("NVIDIA GPU not detected")

if subprocess.run(
    ["nvidia-smi", "-L"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
).returncode != 0:
    raise RuntimeError("NVIDIA GPU not detected")

GPU_NAME = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0]

print("GPU:", GPU_NAME)


GPU: NVIDIA GeForce RTX 4070 SUPER


In [2]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
BASE_DIR = PROJECT_DIR / "artifacts" / "base_model"
V26_DIR = PROJECT_DIR / "artifacts" / "solution_v26"
ARTIFACT_ROOT = PROJECT_DIR / "artifacts" / "solution_v27"
HYBRID_DIR = V26_DIR / "_cache" / "hybrid"
BASE_FRAME_DIR = BASE_DIR / "_cache" / "model_frames"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

V9_CANONICAL_POOL = [
    date(2025, 2, 13),
    date(2025, 3, 26),
    date(2025, 5, 7),
    date(2025, 6, 18),
    date(2025, 7, 30),
    date(2025, 9, 10),
    date(2025, 10, 22),
    date(2025, 11, 12),
    date(2025, 12, 3),
    date(2025, 12, 15),
    date(2026, 1, 14),
]

FIXED_RECENT_WEIGHT = 0.46619
FIXED_HURDLE_WEIGHT = 0.53381
CONTROL_GAMMA = 2.0
LOW_VALUE_BLOCK_UNTIL_RANK = 0.60
LOW_VALUE_FULL_V12_FROM_RANK = 0.90

SAMPLE_PATH = next(
    p for p in [
        DATA_DIR / "sample_submit.csv",
        DATA_DIR / "sample_submission.csv",
        PROJECT_DIR / "sample_submit.csv",
        PROJECT_DIR / "sample_submission.csv",
    ]
    if p.is_file()
)

submit_template = pl.read_csv(SAMPLE_PATH).select("user_id")
assert submit_template.height == EXPECTED_USERS

COMPONENTS_PATH = V26_DIR / "final_components.parquet"
CONTRACT_PATH = V26_DIR / "feature_contract.json"

if not COMPONENTS_PATH.is_file() or not CONTRACT_PATH.is_file():
    raise FileNotFoundError("Run 04_solution_v26.ipynb first")

components = (
    submit_template.with_row_index("__row")
    .join(
        pl.read_parquet(COMPONENTS_PATH),
        on="user_id",
        how="left",
        validate="1:1",
    )
    .sort("__row")
    .drop("__row")
)

contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
BASE_FEATURE_COLS = contract["base_feature_cols"]
A_FEATURE_COLS = contract["A_feature_cols"]
RECENT_FEATURE_COLS = BASE_FEATURE_COLS + [
    "aux_mean_future_to_ord_30d",
    "aux_mean_orders_per_order_day",
]

for anchor in V9_CANONICAL_POOL:
    if not (
        HYBRID_DIR / f"hybrid_{anchor}_1.parquet"
    ).is_file():
        raise FileNotFoundError(
            f"Missing V26 hybrid cache for {anchor}"
        )

if not (
    HYBRID_DIR / "hybrid_2026-02-13_0.parquet"
).is_file():
    raise FileNotFoundError("Missing V26 final hybrid cache")

for anchor in V9_CANONICAL_POOL:
    p = BASE_FRAME_DIR / f"model_frame_{anchor}_target1.parquet"
    if not p.is_file():
        raise FileNotFoundError(p)

final_base_path = BASE_FRAME_DIR / "model_frame_2026-02-13_target0.parquet"
if not final_base_path.is_file():
    raise FileNotFoundError(final_base_path)


## Модель

In [3]:
def _matrix(
    frame: pl.DataFrame,
    cols: list[str],
    mask: np.ndarray | None = None,
) -> np.ndarray:
    part = frame
    if mask is not None:
        part = frame.filter(
            pl.Series("__mask", np.asarray(mask, dtype=bool))
        )
    return part.select(cols).to_numpy().astype(np.float32, copy=False)

def recency_weights(frame: pl.DataFrame, decay_days: float) -> np.ndarray:
    anchor_days = frame["anchor_date"].cast(pl.Int32).to_numpy()
    newest = int(np.max(anchor_days))
    age = newest - anchor_days
    w = np.exp(-age / decay_days).astype(np.float32)
    return w / max(float(w.mean()), 1e-12)

def temporal_weights(
    frame: pl.DataFrame,
    anchors: list[date],
    decay_days: float,
) -> np.ndarray:
    newest = max(anchors)
    newest_day = (newest - date(1970, 1, 1)).days
    anchor_days = frame["anchor_date"].cast(pl.Int32).to_numpy()
    raw = np.exp(
        -(newest_day - anchor_days) / decay_days
    ).astype(np.float32)

    norm = float(np.mean([
        math.exp(-(newest - a).days / decay_days)
        for a in anchors
    ]))
    return raw / max(norm, 1e-12)

def params(cfg: dict, classifier: bool = False) -> dict:
    p = {
        "iterations": int(cfg["iterations"]),
        "loss_function": "Logloss" if classifier else "RMSE",
        "eval_metric": "Logloss" if classifier else "RMSE",
        "depth": int(cfg["depth"]),
        "learning_rate": float(cfg["learning_rate"]),
        "l2_leaf_reg": float(cfg["l2_leaf_reg"]),
        "random_strength": float(cfg["random_strength"]),
        "grow_policy": cfg.get("grow_policy", "SymmetricTree"),
        "score_function": cfg.get("score_function", "Cosine"),
        "border_count": int(cfg.get("border_count", 254)),
        "boosting_type": cfg.get("boosting_type", "Plain"),
        "random_seed": int(cfg["random_seed"]),
        "allow_writing_files": False,
        "task_type": "GPU",
        "devices": DEVICES,
        "verbose": False,
        "bootstrap_type": cfg.get("bootstrap_type", "Bayesian"),
        "leaf_estimation_method": cfg.get(
            "leaf_estimation_method",
            "Newton" if classifier else "Gradient",
        ),
        "leaf_estimation_iterations": int(
            cfg.get(
                "leaf_estimation_iterations",
                10 if classifier else 1,
            )
        ),
    }

    if p["bootstrap_type"] == "Bayesian":
        p["bagging_temperature"] = float(
            cfg.get("bagging_temperature", 0.6)
        )
    elif p["bootstrap_type"] in {"Bernoulli", "Poisson"}:
        p["subsample"] = float(cfg.get("subsample", 0.8))

    return p

def load_base(anchor: date, with_target: bool) -> pl.DataFrame:
    p = BASE_FRAME_DIR / f"model_frame_{anchor}_target{int(with_target)}.parquet"
    out = pl.read_parquet(p)
    if out.schema["anchor_date"] != pl.Date:
        out = out.with_columns(pl.col("anchor_date").cast(pl.Date))
    return out

def load_hybrid(anchor: date, with_target: bool) -> pl.DataFrame:
    return pl.read_parquet(
        HYBRID_DIR / f"hybrid_{anchor}_{int(with_target)}.parquet"
    )


In [4]:
A0_CFG = {
    "iterations": 1784,
    "depth": 8,
    "learning_rate": 0.02020453138039556,
    "l2_leaf_reg": 3.7296561917972872,
    "random_strength": 0.11299833172000943,
    "bootstrap_type": "No",
    "border_count": 254,
    "random_seed": 1043,
    "grow_policy": "SymmetricTree",
    "score_function": "NewtonL2",
    "leaf_estimation_method": "Gradient",
    "leaf_estimation_iterations": 3,
    "boosting_type": "Plain",
    "decay_days": 180.0,
}

A_RSDN_CFG = dict(A0_CFG)
A_RSDN_CFG["random_strength"] = max(
    0.01,
    A0_CFG["random_strength"] * 0.5,
)

RECENT15_CFG = {
    "iterations": 1675,
    "depth": 7,
    "learning_rate": 0.029568343923953,
    "l2_leaf_reg": 4.967674966193398,
    "random_strength": 0.473479601740837,
    "bootstrap_type": "Poisson",
    "subsample": 0.8266831037495286,
    "border_count": 254,
    "random_seed": 143,
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "leaf_estimation_method": "Newton",
    "leaf_estimation_iterations": 1,
    "boosting_type": "Plain",
    "decay_days": 180.0,
}

OCC_CFG = {
    "iterations": 820,
    "depth": 7,
    "learning_rate": 0.04,
    "l2_leaf_reg": 8.0,
    "random_strength": 0.3,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.7,
    "border_count": 254,
    "random_seed": 243,
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "leaf_estimation_method": "Newton",
    "leaf_estimation_iterations": 10,
    "boosting_type": "Plain",
    "decay_days": 180.0,
}


In [5]:
hybrid_train = pl.concat(
    [
        load_hybrid(anchor, True)
        for anchor in V9_CANONICAL_POOL
    ],
    how="vertical_relaxed",
    rechunk=False,
)
hybrid_final = load_hybrid(date(2026, 2, 13), False)

y = hybrid_train["target"].to_numpy().astype(np.float64)
days = hybrid_train["future_spend_days"].to_numpy().astype(np.float64)
pos = y > 0

Xtr = _matrix(hybrid_train, A_FEATURE_COLS, pos)
Xva = _matrix(hybrid_final, A_FEATURE_COLS)
sw = recency_weights(
    hybrid_train,
    A_RSDN_CFG["decay_days"],
)[pos]

model_a = CatBoostRegressor(**params(A_RSDN_CFG))
model_a.fit(
    Xtr,
    np.log(days[pos]),
    sample_weight=sw,
    verbose=False,
)
pred_a_rsdown = model_a.predict(Xva).astype(np.float64)

del model_a, Xtr, Xva, sw, hybrid_train, hybrid_final
gc.collect()

base_train = pl.concat(
    [
        load_base(anchor, True)
        for anchor in V9_CANONICAL_POOL
    ],
    how="vertical_relaxed",
    rechunk=False,
)
base_final = load_base(date(2026, 2, 13), False)

recent_anchors = V9_CANONICAL_POOL[-4:]
recent_train = base_train.filter(
    pl.col("anchor_date").is_in(recent_anchors)
)

Xtr = _matrix(recent_train, RECENT_FEATURE_COLS)
Xva = _matrix(base_final, RECENT_FEATURE_COLS)
yr = recent_train["target"].to_numpy().astype(np.float64)
sw = temporal_weights(
    recent_train,
    V9_CANONICAL_POOL,
    RECENT15_CFG["decay_days"],
)

recent_model = CatBoostRegressor(**params(RECENT15_CFG))
recent_model.fit(
    Xtr,
    np.log1p(np.clip(yr, 0.0, None)),
    sample_weight=sw,
    verbose=False,
)
pred_recent15 = np.clip(
    recent_model.predict(Xva).astype(np.float64),
    0.0,
    None,
)

del recent_model, Xtr, Xva, yr, sw, recent_train
gc.collect()

yb = base_train["target"].to_numpy().astype(np.float64)
occ_target = (yb > 0).astype(np.int8)

Xtr = _matrix(base_train, BASE_FEATURE_COLS + ["aux_mean_p_any_order"])
Xva = _matrix(base_final, BASE_FEATURE_COLS + ["aux_mean_p_any_order"])
sw = recency_weights(base_train, OCC_CFG["decay_days"])

occ_model = CatBoostClassifier(**params(OCC_CFG, classifier=True))
occ_model.fit(
    Xtr,
    occ_target,
    sample_weight=sw,
    verbose=False,
)
pred_occ_control = np.clip(
    occ_model.predict_proba(Xva)[:, 1].astype(np.float64),
    0.0,
    1.0,
)

del occ_model, Xtr, Xva, sw, base_train, base_final
gc.collect()

assert np.isfinite(pred_a_rsdown).all()
assert np.isfinite(pred_recent15).all()
assert np.isfinite(pred_occ_control).all()


## Сабмит

In [6]:
def mix(a: np.ndarray, b: np.ndarray, weight: float) -> np.ndarray:
    return np.asarray(a) + weight * (
        np.asarray(b) - np.asarray(a)
    )

def rank_gate(base: np.ndarray) -> np.ndarray:
    v = np.asarray(base, dtype=np.float64)
    sorted_v = np.sort(v)
    rank = np.searchsorted(
        sorted_v,
        v,
        side="right",
    ).astype(np.float64) / len(v)

    x = (
        rank - LOW_VALUE_BLOCK_UNTIL_RANK
    ) / (
        LOW_VALUE_FULL_V12_FROM_RANK
        - LOW_VALUE_BLOCK_UNTIL_RANK
    )
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)

def compose(
    recent: np.ndarray,
    occurrence: np.ndarray,
    positive: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
) -> np.ndarray:
    recent = np.asarray(recent, dtype=np.float64)
    occurrence = np.clip(
        np.asarray(occurrence, dtype=np.float64),
        0.0,
        1.0,
    )
    positive = np.clip(
        np.asarray(positive, dtype=np.float64),
        0.0,
        None,
    )

    base = np.clip(
        FIXED_RECENT_WEIGHT * recent
        + FIXED_HURDLE_WEIGHT * occurrence * positive,
        0.0,
        None,
    )

    severity = np.clip(
        np.asarray(pred_a, dtype=np.float64)
        + np.asarray(pred_b, dtype=np.float64),
        0.0,
        None,
    )

    full = np.clip(
        FIXED_RECENT_WEIGHT * recent
        + FIXED_HURDLE_WEIGHT * occurrence * severity,
        0.0,
        None,
    )

    delta = full - base
    gate = occurrence ** CONTROL_GAMMA * rank_gate(base)
    return np.clip(
        base + np.where(delta > 0.0, gate * delta, delta),
        0.0,
        None,
    )

recent = mix(
    components["recent_exact"].to_numpy(),
    pred_recent15,
    0.45,
)

occurrence = np.clip(
    mix(
        components["occurrence_exact"].to_numpy(),
        pred_occ_control,
        0.75,
    ),
    0.0,
    1.0,
)

positive = components["positive_pos09"].to_numpy().astype(np.float64)
pred_b = components["pred_b_j065"].to_numpy().astype(np.float64)

pred_log = compose(
    recent,
    occurrence,
    positive,
    pred_a_rsdown,
    pred_b,
)

submission = pl.DataFrame({
    "user_id": submit_template["user_id"],
    "predict": np.clip(
        np.expm1(np.clip(pred_log, 0.0, 30.0)),
        0.0,
        None,
    ),
})

SUBMISSION_PATH = ARTIFACT_ROOT / "submission_V27_172071bcb3.csv"
submission.write_csv(SUBMISSION_PATH)

pl.DataFrame({
    "user_id": submit_template["user_id"],
    "recent_v27": recent,
    "occurrence_v27": occurrence,
    "positive_v27": positive,
    "pred_a_v27": pred_a_rsdown,
    "pred_b_v27": pred_b,
    "v27_pred_log": pred_log,
}).write_parquet(
    ARTIFACT_ROOT / "final_components.parquet",
    compression="zstd",
)

run = {
    "solution": "V27_172071bcb3",
    "parent": "J065_e96d822fa5",
    "A": "V27_A_RSDN__47e6bdb3cd",
    "B": "B_REPRK20__31a21f6262",
    "recent": {
        "source": "RECENT_15__96f7ca6405",
        "weight": 0.45,
    },
    "occurrence": {
        "source": "OCC_PARAM_CONTROL__283268adbf",
        "weight": 0.75,
    },
    "positive": {
        "source": "POS_09__59b5afea6e",
        "weight": 1.0,
    },
}
(ARTIFACT_ROOT / "config.json").write_text(
    json.dumps(run, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

assert submission.height == EXPECTED_USERS
assert submission["predict"].null_count() == 0
assert np.isfinite(submission["predict"].to_numpy()).all()
assert float(submission["predict"].min()) >= 0.0

print("Submission:", SUBMISSION_PATH.resolve())


Submission: D:\baboteek_holmotech\artifacts\solution_v27\submission_V27_172071bcb3.csv
